# TMJDLC — Google Colab Download Test

Run each cell in order. This notebook tests the repository's current MissAV provider and native asynchronous HLS downloader. Stream URLs are temporary: resolution and downloading happen in the same Colab runtime/session, and temporary media URLs are not written to the report.

**Runtime:** Python 3.12 recommended. Keep this notebook and the downloader in the same runtime; do not switch networks or proxies mid-download.

In [ ]:
# Install TMJDLC from the latest main branch
!git clone --depth 1 https://github.com/Telebotfaroff/TMJDLC.git
%cd TMJDLC
!python -m pip install --upgrade pip
!python -m pip install -e .
!sudo apt-get update -qq && sudo apt-get install -y ffmpeg

## 1. Configure the test
Enter the original video page URL, not a previously resolved stream URL.

In [ ]:
#@title Download settings
VIDEO_URL = "" #@param {type:"string"}
QUALITY = "best" #@param ["best", "half", "worst", "1080", "720", "480", "360"]
OUTPUT_DIR = "/content/tmjdlc-downloads" #@param {type:"string"}

if not VIDEO_URL.strip():
    raise ValueError("Set VIDEO_URL to the original video page URL first.")
from pathlib import Path
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")
print(f"Requested quality: {QUALITY}")

## 2. Fetch metadata and download
The provider resolves the current playback information and starts the download inside the same runtime. Do not copy or reuse a temporary stream URL.

In [ ]:
import asyncio
import json
import time
from pathlib import Path
from tmjdlc.providers.missav import MissAVProvider

async def run_download_test():
    started = time.monotonic()
    async with MissAVProvider() as provider:
        print("Fetching video metadata...")
        video = await provider.get_video(VIDEO_URL.strip())
        metadata = provider.metadata(video)
        print(json.dumps(metadata, ensure_ascii=False, indent=2))
        print("\nStarting download in this same runtime...")
        await provider.download(video, OUTPUT_DIR, quality=QUALITY)
    elapsed = time.monotonic() - started
    files = [p for p in Path(OUTPUT_DIR).rglob("*") if p.is_file()]
    if not files:
        raise RuntimeError("Downloader returned but no output file was found.")
    newest = max(files, key=lambda p: p.stat().st_mtime)
    size = newest.stat().st_size
    if size <= 0:
        raise RuntimeError(f"Downloaded file is empty: {newest.name}")
    result = {
        "status": "downloaded",
        "filename": newest.name,
        "size_bytes": size,
        "size_mib": round(size / (1024 ** 2), 2),
        "elapsed_seconds": round(elapsed, 2),
        "average_mib_per_second": round(size / (1024 ** 2) / elapsed, 3) if elapsed else None,
        "quality_requested": QUALITY,
        "output_path": str(newest),
    }
    print("\nDOWNLOAD RESULT")
    print(json.dumps(result, indent=2))
    return newest, result

downloaded_path, download_result = await run_download_test()

## 3. Validate the downloaded file
FFprobe checks the container, streams, duration and detected resolution.

In [ ]:
import subprocess

probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries",
     "format=format_name,duration,size:stream=codec_type,codec_name,width,height",
     "-of", "json", str(downloaded_path)],
    capture_output=True, text=True,
)
if probe.returncode != 0:
    print("FFprobe validation failed:")
    print(probe.stderr[-4000:])
    raise RuntimeError("Downloaded file could not be validated by FFprobe.")
probe_data = json.loads(probe.stdout)
print(json.dumps(probe_data, indent=2))

## 4. Download the file to your device
This downloads the completed file from the Colab runtime to your browser/device. Large files may take time.

In [ ]:
from google.colab import files
files.download(str(downloaded_path))

## 5. Share test results
If anything fails, share the full traceback and the printed `DOWNLOAD RESULT` / FFprobe output. Do not share cookies, authorization headers, signed stream URLs, or session credentials. The notebook intentionally reports the original page metadata and local file metrics, not temporary media URLs.